# Task 1 — Extract

Retrieves tender records from the Etimad public tenders portal and saves them,
unmodified, to `data/raw/`. Cleaning, translation, and agency classification are
handled in Task 2 (`02_profile_clean.ipynb`).

**Data source:** internal JSON endpoint used by the Etimad public listing page
(see `README.md` → Source Definition for endpoint details, auth, rate limits,
and licence notes).


## 1. Imports

In [1]:
import requests
import json
import time
from datetime import date
from pathlib import Path


## 2. Configuration

In [2]:
BASE_URL = "https://tenders.etimad.sa/Tender/AllSupplierTendersForVisitorAsync"

PAGE_SIZE = 100
PAGES_TO_SCRAPE = 20
PUBLISH_DATE_ID = 5  # matches the default filter applied on the public listing page

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
TODAY = date.today().isoformat()

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://tenders.etimad.sa/Tender/AllTendersForVisitor?PageNumber=1",
}


## 3. Fetch a single page

Requests can return `429 Too Many Requests` under rapid successive calls.
Retries with exponential backoff on `429`, and a short backoff on generic
connection errors.


In [3]:
def fetch_page(page_number, page_size=PAGE_SIZE, max_retries=4):
    params = {
        "PageSize": page_size,
        "PublishDateId": PUBLISH_DATE_ID,
        "pageNumber": page_number,
    }

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=20)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.HTTPError:
            if response.status_code == 429:
                wait_time = 5 * attempt
                print(f"Page {page_number}: 429 Too Many Requests — retrying in {wait_time}s "
                      f"(attempt {attempt}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"Page {page_number}: HTTP {response.status_code} — aborting")
                raise
        except requests.exceptions.RequestException as e:
            print(f"Page {page_number}: connection error ({e}) — retrying in 5s")
            time.sleep(5)

    raise RuntimeError(f"Failed to fetch page {page_number} after {max_retries} attempts")


## 4. Inspect response shape

Run once to confirm the response structure before building the extraction loop.
Prints the top-level type, the key holding the tender list, and one sample record.


In [4]:
sample = fetch_page(page_number=1, page_size=5)

items_preview = []

if isinstance(sample, list):
    items_preview = sample
elif isinstance(sample, dict):
    print("Top-level keys:", list(sample.keys()))
    for key in ["data", "Data", "result", "Result", "items", "Items", "tenders"]:
        if key in sample and isinstance(sample[key], list):
            items_preview = sample[key]
            print(f"Tender list found under key: '{key}' ({len(items_preview)} items)")
            break

if items_preview:
    print("\nFields on a sample record:")
    print(list(items_preview[0].keys()))


Top-level keys: ['data', 'totalCount', 'pageSize', 'queryString', 'currentPage']
Tender list found under key: 'data' (5 items)

Fields on a sample record:
['tenderId', 'referenceNumber', 'tenderName', 'tenderNumber', 'multipleSearch', 'agencyCode', 'branchId', 'branchName', 'agencyName', 'tenderIdString', 'tenderStatusId', 'tenderStatusIdString', 'tenderStatusName', 'tenderTypeId', 'tenderTypeName', 'technicalOrganizationId', 'condetionalBookletPrice', 'createdAt', 'lastEnqueriesDate', 'lastOfferPresentationDate', 'offersOpeningDate', 'lastEnqueriesDateHijri', 'offersOpeningDateHijri', 'lastOfferPresentationDateHijri', 'insideKSA', 'tenderActivityName', 'tenderActivityNameList', 'tenderActivityId', 'submitionDate', 'financialFees', 'invitationCost', 'buyingCost', 'hasInvitations', 'remainingDays', 'remainingHours', 'remainingMins', 'currentDate', 'currentDateTime', 'currentTime', 'isUGRP', 'ugrpRfxUrl', 'ugrpRFXResponseURL']


## 5. Extract the tender list from a response

`RESPONSE_LIST_KEY` is confirmed from the inspection above: the Etimad endpoint
wraps the tender list under `"data"`, alongside `"totalCount"`, `"pageSize"`,
and `"currentPage"`.


In [5]:
RESPONSE_LIST_KEY = "data"


def extract_items(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and RESPONSE_LIST_KEY:
        return payload.get(RESPONSE_LIST_KEY, [])
    return []


def get_total_count(payload):
    if isinstance(payload, dict):
        return payload.get("totalCount")
    return None


## 6. Run the extraction

Iterates through pages and deduplicates on `tenderId`, which is a stable
unique identifier returned by the API. Stops early if a page returns no items.


In [6]:
def run_extraction(pages_to_scrape):
    all_items = []
    seen_ids = set()

    for page_number in range(1, pages_to_scrape + 1):
        payload = fetch_page(page_number)
        items = extract_items(payload)

        if page_number == 1:
            total_available = get_total_count(payload)
            if total_available is not None:
                print(f"Total tenders available on the portal: {total_available}\n")

        if not items:
            print(f"Page {page_number}: no items returned — stopping")
            break

        new_count = 0
        for item in items:
            tender_id = item.get("tenderId")
            if tender_id is not None and tender_id not in seen_ids:
                seen_ids.add(tender_id)
                all_items.append(item)
                new_count += 1

        print(f"Page {page_number}: {len(items)} items, {new_count} new")
        time.sleep(3)

    return all_items


records = run_extraction(PAGES_TO_SCRAPE)
print(f"\nTotal unique records extracted: {len(records)}")


Total tenders available on the portal: 7588

Page 1: 24 items, 24 new
Page 2: 24 items, 24 new
Page 3: 24 items, 24 new
Page 4: 24 items, 24 new
Page 5: 24 items, 24 new
Page 6: 24 items, 24 new
Page 7: 24 items, 24 new
Page 8: 24 items, 24 new
Page 9: 24 items, 24 new
Page 10: 24 items, 24 new
Page 11: 24 items, 24 new
Page 12: 24 items, 24 new
Page 13: 24 items, 24 new
Page 14: 24 items, 24 new
Page 15: 24 items, 24 new
Page 16: 24 items, 24 new
Page 17: 24 items, 24 new
Page 18: 24 items, 24 new
Page 19: 24 items, 24 new
Page 20: 24 items, 24 new

Total unique records extracted: 480


## 7. Save raw output

Records are saved exactly as returned by the API — no field selection, renaming, or cleaning at this stage.

In [7]:
output_path = RAW_DIR / f"etimad_all_tenders_{TODAY}.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Saved to: {output_path}")


Saved to: ../data/raw/etimad_all_tenders_2026-09-10.json


## 8. Source diversity summary

Informational only. Final agency classification (mapping raw agency names to canonical sources) happens in Task 2.

In [8]:
AGENCY_FIELD_NAME = "agencyName"

distinct_agencies = sorted(set(
    r.get(AGENCY_FIELD_NAME) for r in records if r.get(AGENCY_FIELD_NAME)
))
distinct_activities = sorted(set(
    r.get("tenderActivityName") for r in records if r.get("tenderActivityName")
))

print(f"Total tenders: {len(records)}")
print(f"Distinct raw agency names: {len(distinct_agencies)}")
print(f"Distinct activity/category names: {len(distinct_activities)}")


Total tenders: 480
Distinct raw agency names: 173
Distinct activity/category names: 53
